# Ollama + FastAPI + Cloudflare Tunnel → Claude Code バックエンド (Colab T4)

Google Colab の無料GPU(T4)上で Ollama を動かし、Anthropic Messages API 互換の `/v1/messages` を
FastAPI で提供し、Cloudflare Tunnel で公開してローカルの Claude Code から使えるようにします。

参考: [gist.github.com/SanjayPG](https://gist.github.com/SanjayPG/f6ae22a06f05be3628df24ab6a03341c) のセットアップを元に、
tool_use対応・count_tokens実装・APIキー認証・タイムアウト延長などを改善しています。

**上から順にセルを実行してください。** 最後のセルでトンネルURLとAPIキーが表示されるので、
ローカルの `start-local-llm.sh` に渡してください。

**注意**: Colab無料枠は概ね90分アイドルで切断されます。長時間使う場合はブラウザタブを開いたままにしてください。

In [ ]:
import secrets

# ==== Configuration ====
# Switch models here by editing this one line.
#   - deepseek-coder-v2:16b : matches the original gist. Its Ollama template may not
#     support OpenAI-style "tools", in which case Claude Code falls back to plain text
#     answers (no file edits / command execution).
#   - qwen2.5-coder:14b     : Ollama tool-calling compatible, fits comfortably in the
#     T4's 15GB VRAM, and is the recommended choice if you want to actually exercise
#     Claude Code's agentic features (see README "tool_use検証手順").
MODEL_NAME = "deepseek-coder-v2:16b"  # alt: "qwen2.5-coder:14b"

API_KEY = secrets.token_hex(16)

print(f"Model:   {MODEL_NAME}")
print(f"API key: {API_KEY}")
print("\nKeep this API key - you'll need it for ./start-local-llm.sh")

## 1. GPUの確認

Colabのランタイムが GPU(できれば T4)になっていることを確認します。
`ランタイム > ランタイムのタイプを変更 > T4 GPU` を選んでから実行してください。

In [ ]:
!nvidia-smi

## 2. 依存関係のインストール

`pciutils` は Ollama インストーラの GPU 検出に必要です。

In [ ]:
!sudo apt-get update -qq && sudo apt-get install -y -qq zstd pciutils
!pip install -q fastapi uvicorn nest_asyncio requests

## 3. Ollama のインストール

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

## 4. Ollama サーバーの起動

`OLLAMA_KEEP_ALIVE=-1` でモデルをメモリに常駐させ、リクエストのたびに再ロードされるのを防ぎます
(デフォルトは5分でアンロードされ、次回リクエストが大幅に遅くなります)。
固定の `sleep` ではなく `/api/version` へのポーリングで起動完了を待ちます。

In [ ]:
import subprocess
import os
import time
import requests

env = os.environ.copy()
env["OLLAMA_KEEP_ALIVE"] = "-1"  # never unload the model between requests

ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
    env=env,
)

for _ in range(30):
    try:
        r = requests.get("http://127.0.0.1:11434/api/version", timeout=2)
        if r.status_code == 200:
            break
    except requests.exceptions.RequestException:
        pass
    time.sleep(1)
else:
    raise RuntimeError("Ollama server did not start in time - check the runtime logs")

print("Ollama server started!")

## 5. モデルの pull

`deepseek-coder-v2:16b` は約10GBのダウンロードがあり、数分かかります。

In [ ]:
!ollama pull {MODEL_NAME}

## 6. モデルの動作確認

In [ ]:
!ollama run {MODEL_NAME} "Write hello world in Python"

## 7. FastAPI サーバー

元gistのコードから以下を改善しています:

- **tool_use対応**: Anthropicの `tools`/`tool_use`/`tool_result` ⇔ OpenAIの `tools`/`tool_calls` を
  双方向変換(streaming含む)。モデルが tools 非対応の場合は自動的に tools を外してリトライします。
- **tool_use テキストフォールバック検出**: Qwen系モデルはOllama経由でも構造化された `tool_calls` を
  返さず、`<tool_call>{"name":...,"arguments":{...}}</tool_call>` のようなプレーンテキストで
  ツール呼び出しを返すことがあります。この形式を検出して `tool_use` ブロックに変換するため、
  Claude Code側で実際にツールが実行されます(単にJSONが表示されるだけの状態を回避)。
  streaming時は tools 指定リクエストのみ応答全体をバッファしてから検出・送出します
  (tools無しリクエストは従来どおり逐次ストリーミング)。
- **`/v1/messages/count_tokens`**: 簡易実装(文字数ベースの概算。CJKは1文字≒1トークン、それ以外は4文字≒1トークン)
- **APIキー認証**: `x-api-key` ヘッダー(または `Authorization: Bearer`)を検証。`/health` のみ認証不要
- **タイムアウト延長**: `timeout=(10, 600)`(接続10秒/読み取り600秒)
- **system配列対応**: Claude Codeが送る `system: [{"type":"text",...}]` 形式に対応
- **モデル名マッピング**: リクエストの `model`(例: `claude-sonnet-4-5`)は無視し、常に `MODEL_NAME` へ転送
- **エラーレスポンス**: Anthropic形式のエラーボディ + 適切なHTTPステータス

In [ ]:
from fastapi import FastAPI, Request, Header, HTTPException, Depends
from fastapi.responses import StreamingResponse, JSONResponse
from typing import Optional
import requests
import uvicorn
import nest_asyncio
import json
import re
import uuid

nest_asyncio.apply()
app = FastAPI()

OLLAMA_URL = "http://127.0.0.1:11434/v1/chat/completions"
REQUEST_TIMEOUT = (10, 600)  # (connect, read) seconds - long read timeout for slow local GPU generation


# ---------- Auth ----------

async def verify_api_key(x_api_key: Optional[str] = Header(None), authorization: Optional[str] = Header(None)):
    if not API_KEY:
        return
    provided = x_api_key
    if not provided and authorization and authorization.lower().startswith("bearer "):
        provided = authorization[7:]
    if provided != API_KEY:
        raise HTTPException(status_code=401, detail="invalid api key")


def anthropic_error_response(status_code, message, err_type="api_error"):
    return JSONResponse(
        status_code=status_code,
        content={"type": "error", "error": {"type": err_type, "message": message}},
    )


# ---------- Anthropic <-> OpenAI message conversion ----------

def system_to_text(system):
    if not system:
        return ""
    if isinstance(system, list):
        return "\n".join(b.get("text", "") for b in system if isinstance(b, dict))
    return system


def convert_anthropic_to_openai_messages(system, messages):
    openai_messages = []
    sys_text = system_to_text(system)
    if sys_text:
        openai_messages.append({"role": "system", "content": sys_text})

    for msg in messages:
        role = msg.get("role")
        content = msg.get("content")

        if isinstance(content, str):
            openai_messages.append({"role": role, "content": content})
            continue
        if not isinstance(content, list):
            continue

        if role == "assistant":
            text_parts = []
            tool_calls = []
            for block in content:
                btype = block.get("type")
                if btype == "text":
                    text_parts.append(block.get("text", ""))
                elif btype == "tool_use":
                    tool_calls.append({
                        "id": block.get("id") or f"call_{uuid.uuid4().hex[:16]}",
                        "type": "function",
                        "function": {
                            "name": block.get("name", ""),
                            "arguments": json.dumps(block.get("input", {})),
                        },
                    })
            entry = {"role": "assistant", "content": "\n".join(text_parts) if text_parts else None}
            if tool_calls:
                entry["tool_calls"] = tool_calls
            openai_messages.append(entry)

        elif role == "user":
            text_parts = []
            tool_messages = []
            for block in content:
                btype = block.get("type")
                if btype == "text":
                    text_parts.append(block.get("text", ""))
                elif btype == "tool_result":
                    tr_content = block.get("content", "")
                    if isinstance(tr_content, list):
                        tr_text = "\n".join(
                            b.get("text", "") for b in tr_content if isinstance(b, dict) and b.get("type") == "text"
                        )
                    else:
                        tr_text = str(tr_content)
                    tool_messages.append({
                        "role": "tool",
                        "tool_call_id": block.get("tool_use_id", ""),
                        "content": tr_text,
                    })
            if text_parts:
                openai_messages.append({"role": "user", "content": "\n".join(text_parts)})
            openai_messages.extend(tool_messages)

        else:
            text = "\n".join(b.get("text", "") for b in content if isinstance(b, dict) and b.get("type") == "text")
            openai_messages.append({"role": role, "content": text})

    return openai_messages


def convert_anthropic_tools_to_openai(tools):
    if not tools:
        return None
    openai_tools = []
    for t in tools:
        openai_tools.append({
            "type": "function",
            "function": {
                "name": t.get("name"),
                "description": t.get("description", ""),
                "parameters": t.get("input_schema", {"type": "object", "properties": {}}),
            },
        })
    return openai_tools


def convert_openai_message_to_anthropic_content(message):
    blocks = []
    content = message.get("content")
    if content:
        blocks.append({"type": "text", "text": content})
    for tc in message.get("tool_calls") or []:
        func = tc.get("function", {})
        try:
            tool_input = json.loads(func.get("arguments") or "{}")
        except json.JSONDecodeError:
            tool_input = {}
        blocks.append({
            "type": "tool_use",
            "id": tc.get("id") or f"call_{uuid.uuid4().hex[:16]}",
            "name": func.get("name", ""),
            "input": tool_input,
        })
    if not blocks:
        blocks.append({"type": "text", "text": ""})
    return blocks


# ---------- fallback: detect tool calls emitted as plain text ----------
#
# Some Ollama/model combinations (e.g. Qwen2.5-Coder) don't reliably populate the
# structured OpenAI "tool_calls" field even when "tools" was passed in the request.
# Instead the model emits its tool call as text, following the Hermes/Qwen convention:
#   <tool_call>
#   {"name": "...", "arguments": {...}}
#   </tool_call>
# or occasionally as a single bare (optionally fenced) JSON object with no surrounding
# text at all. We detect these patterns and convert them into real Anthropic tool_use
# blocks so Claude Code actually executes the tool instead of just printing the JSON.

TOOL_CALL_TAG_RE = re.compile(r"<tool_call>\s*(\{.*?\})\s*</tool_call>", re.DOTALL)
BARE_JSON_FENCE_RE = re.compile(r"^```(?:json)?\s*(\{.*\})\s*```$", re.DOTALL)


def _try_parse_fallback_tool_call(json_str, valid_tool_names):
    try:
        obj = json.loads(json_str)
    except (json.JSONDecodeError, TypeError):
        return None
    if not isinstance(obj, dict):
        return None
    name = obj.get("name")
    if name not in valid_tool_names:
        return None
    args = obj.get("arguments", obj.get("input", {}))
    if not isinstance(args, dict):
        args = {}
    return {"type": "tool_use", "id": f"toolu_{uuid.uuid4().hex[:24]}", "name": name, "input": args}


def extract_fallback_tool_calls(text, valid_tool_names):
    """Returns (leftover_text, [tool_use_block, ...])."""
    if not text or not valid_tool_names:
        return text, []

    tag_matches = list(TOOL_CALL_TAG_RE.finditer(text))
    if tag_matches:
        pieces = []
        calls = []
        last_end = 0
        for m in tag_matches:
            pieces.append(text[last_end:m.start()])
            last_end = m.end()
            parsed = _try_parse_fallback_tool_call(m.group(1), valid_tool_names)
            if parsed:
                calls.append(parsed)
            else:
                pieces.append(m.group(0))
        pieces.append(text[last_end:])
        if calls:
            return "".join(pieces).strip(), calls
        return text, []

    # No <tool_call> tags - check if the whole message is one bare JSON tool call,
    # optionally wrapped in a markdown code fence.
    stripped = text.strip()
    fence_match = BARE_JSON_FENCE_RE.match(stripped)
    candidate = fence_match.group(1) if fence_match else stripped
    parsed = _try_parse_fallback_tool_call(candidate, valid_tool_names)
    if parsed:
        return "", [parsed]

    return text, []


def apply_fallback_tool_detection(content_blocks, anthropic_tools):
    """Post-process content blocks: turn text-encoded tool calls into tool_use blocks."""
    if not anthropic_tools:
        return content_blocks
    valid_names = {t.get("name") for t in anthropic_tools if t.get("name")}
    if not valid_names:
        return content_blocks

    new_blocks = []
    for block in content_blocks:
        if block["type"] != "text":
            new_blocks.append(block)
            continue
        leftover, calls = extract_fallback_tool_calls(block["text"], valid_names)
        if not calls:
            new_blocks.append(block)
            continue
        if leftover:
            new_blocks.append({"type": "text", "text": leftover})
        new_blocks.extend(calls)
    return new_blocks


# ---------- token estimation (for /v1/messages/count_tokens) ----------

def estimate_tokens(text):
    if not text:
        return 0
    cjk = 0
    other = 0
    for ch in text:
        code = ord(ch)
        if (0x3040 <= code <= 0x30FF) or (0x4E00 <= code <= 0x9FFF) or (0xAC00 <= code <= 0xD7A3):
            cjk += 1
        else:
            other += 1
    return cjk + max(0, (other + 3) // 4)


# ---------- Ollama call helpers ----------

def call_ollama(payload):
    return requests.post(OLLAMA_URL, json=payload, timeout=REQUEST_TIMEOUT)


def looks_like_tools_unsupported(resp):
    if resp.status_code == 200:
        return False
    try:
        detail = resp.json()
    except Exception:
        detail = {"error": resp.text}
    msg = json.dumps(detail).lower()
    return "tool" in msg and ("not support" in msg or "unsupported" in msg or "does not support" in msg)


# ---------- OpenAI-compatible endpoints (kept for curl/debugging) ----------

@app.post("/chat")
@app.post("/v1/chat/completions")
async def chat(request: Request, _: None = Depends(verify_api_key)):
    """OpenAI-compatible passthrough to Ollama, for curl-based debugging."""
    try:
        body = await request.json()
    except Exception:
        return anthropic_error_response(400, "invalid JSON body", "invalid_request_error")

    payload = {
        "model": MODEL_NAME,
        "messages": body.get("messages", []),
        "stream": body.get("stream", False),
    }
    if body.get("tools"):
        payload["tools"] = body["tools"]

    if payload["stream"]:
        def generate():
            resp = requests.post(OLLAMA_URL, json=payload, stream=True, timeout=REQUEST_TIMEOUT)
            for line in resp.iter_lines():
                if line:
                    yield line + b"\n"
        return StreamingResponse(generate(), media_type="text/event-stream")

    try:
        resp = call_ollama(payload)
        if resp.status_code == 200:
            return resp.json()
        return JSONResponse(status_code=502, content={"error": "LLM failed", "status": resp.status_code, "details": resp.text})
    except requests.exceptions.RequestException as e:
        return JSONResponse(status_code=502, content={"error": str(e)})


# ---------- Anthropic-compatible endpoint ----------

@app.post("/v1/messages")
async def anthropic_messages(request: Request, _: None = Depends(verify_api_key)):
    try:
        body = await request.json()
    except Exception:
        return anthropic_error_response(400, "invalid JSON body", "invalid_request_error")

    messages = body.get("messages", [])
    system = body.get("system", "")
    max_tokens = body.get("max_tokens", 4096)
    stream = bool(body.get("stream", False))
    anthropic_tools = body.get("tools")
    temperature = body.get("temperature")

    openai_messages = convert_anthropic_to_openai_messages(system, messages)
    openai_tools = convert_anthropic_tools_to_openai(anthropic_tools)

    payload = {
        "model": MODEL_NAME,
        "messages": openai_messages,
        "max_tokens": max_tokens,
        "stream": stream,
    }
    if openai_tools:
        payload["tools"] = openai_tools
    if temperature is not None:
        payload["temperature"] = temperature

    response_model = body.get("model", MODEL_NAME)

    if stream:
        return StreamingResponse(
            stream_anthropic_response(payload, response_model, anthropic_tools),
            media_type="text/event-stream",
        )

    try:
        resp = call_ollama(payload)
        if looks_like_tools_unsupported(resp) and "tools" in payload:
            print(f"[warn] model {MODEL_NAME} rejected tools, retrying without tools")
            payload = dict(payload)
            payload.pop("tools")
            resp = call_ollama(payload)
        if resp.status_code != 200:
            return anthropic_error_response(502, f"Ollama error: {resp.text}", "api_error")
        result = resp.json()
    except requests.exceptions.RequestException as e:
        return anthropic_error_response(502, f"Failed to reach Ollama: {e}", "api_error")

    choice = result["choices"][0]
    message = choice.get("message", {})
    content_blocks = convert_openai_message_to_anthropic_content(message)
    content_blocks = apply_fallback_tool_detection(content_blocks, anthropic_tools)
    has_tool_use = any(b["type"] == "tool_use" for b in content_blocks)
    finish_reason = choice.get("finish_reason")
    if has_tool_use:
        stop_reason = "tool_use"
    elif finish_reason == "length":
        stop_reason = "max_tokens"
    else:
        stop_reason = "end_turn"

    usage = result.get("usage", {})
    return {
        "id": f"msg_{uuid.uuid4().hex[:24]}",
        "type": "message",
        "role": "assistant",
        "content": content_blocks,
        "model": response_model,
        "stop_reason": stop_reason,
        "stop_sequence": None,
        "usage": {
            "input_tokens": usage.get("prompt_tokens", 0),
            "output_tokens": usage.get("completion_tokens", 0),
        },
    }


def sse(event, data):
    return f"event: {event}\ndata: {json.dumps(data)}\n\n"


def consume_ollama_stream_to_message(resp):
    """Fully drain an Ollama streaming response into a single OpenAI-style
    message dict ({"content": ..., "tool_calls": [...]}) plus finish_reason."""
    text_parts = []
    tool_calls_acc = {}  # openai tool_call index -> {"id", "name", "arguments"}
    finish_reason = None

    for line in resp.iter_lines():
        if not line:
            continue
        line_str = line.decode("utf-8", errors="ignore")
        if not line_str.startswith("data: "):
            continue
        data_str = line_str[6:]
        if data_str.strip() == "[DONE]":
            break
        try:
            chunk = json.loads(data_str)
        except json.JSONDecodeError:
            continue
        choices = chunk.get("choices") or []
        if not choices:
            continue
        choice = choices[0]
        delta = choice.get("delta", {})
        if choice.get("finish_reason"):
            finish_reason = choice.get("finish_reason")

        if delta.get("content"):
            text_parts.append(delta["content"])

        for tc in delta.get("tool_calls") or []:
            idx = tc.get("index", 0)
            acc = tool_calls_acc.setdefault(idx, {"id": None, "name": "", "arguments": ""})
            if tc.get("id"):
                acc["id"] = tc["id"]
            func = tc.get("function") or {}
            if func.get("name"):
                acc["name"] += func["name"]
            if func.get("arguments"):
                acc["arguments"] += func["arguments"]

    message = {"content": "".join(text_parts) or None}
    if tool_calls_acc:
        message["tool_calls"] = [
            {
                "id": acc["id"] or f"call_{uuid.uuid4().hex[:16]}",
                "type": "function",
                "function": {"name": acc["name"], "arguments": acc["arguments"]},
            }
            for acc in tool_calls_acc.values()
        ]
    return message, finish_reason


def emit_message_events(message_id, response_model, content_blocks, stop_reason):
    """Emit a full Anthropic SSE event sequence for an already-complete set of
    content blocks (used for the buffered tools-enabled streaming path)."""
    yield sse("message_start", {
        "type": "message_start",
        "message": {
            "id": message_id, "type": "message", "role": "assistant",
            "content": [], "model": response_model,
            "usage": {"input_tokens": 0, "output_tokens": 0},
        },
    })

    for index, block in enumerate(content_blocks):
        if block["type"] == "text":
            yield sse("content_block_start", {
                "type": "content_block_start", "index": index,
                "content_block": {"type": "text", "text": ""},
            })
            yield sse("content_block_delta", {
                "type": "content_block_delta", "index": index,
                "delta": {"type": "text_delta", "text": block["text"]},
            })
        else:
            yield sse("content_block_start", {
                "type": "content_block_start", "index": index,
                "content_block": {"type": "tool_use", "id": block["id"], "name": block["name"], "input": {}},
            })
            yield sse("content_block_delta", {
                "type": "content_block_delta", "index": index,
                "delta": {"type": "input_json_delta", "partial_json": json.dumps(block["input"])},
            })
        yield sse("content_block_stop", {"type": "content_block_stop", "index": index})

    yield sse("message_delta", {
        "type": "message_delta",
        "delta": {"stop_reason": stop_reason, "stop_sequence": None},
        "usage": {"output_tokens": 0},
    })
    yield sse("message_stop", {"type": "message_stop"})


def stream_anthropic_response(payload, response_model, anthropic_tools):
    message_id = f"msg_{uuid.uuid4().hex[:24]}"

    resp = requests.post(OLLAMA_URL, json=payload, stream=True, timeout=REQUEST_TIMEOUT)
    if resp.status_code != 200 and "tools" in payload:
        print(f"[warn] model {MODEL_NAME} rejected tools (streaming), retrying without tools")
        payload = dict(payload)
        payload.pop("tools")
        anthropic_tools = None
        resp = requests.post(OLLAMA_URL, json=payload, stream=True, timeout=REQUEST_TIMEOUT)

    if resp.status_code != 200:
        yield sse("error", {"type": "error", "error": {"type": "api_error", "message": f"Ollama error: {resp.text}"}})
        return

    if anthropic_tools:
        # Buffer the whole response instead of forwarding deltas as they arrive.
        # This lets us reuse the exact same tool_calls + text-fallback detection
        # logic as the non-streaming path, which matters because models like
        # Qwen2.5-Coder don't reliably populate the structured tool_calls field -
        # detecting that requires seeing the complete text first.
        message, finish_reason = consume_ollama_stream_to_message(resp)
        content_blocks = convert_openai_message_to_anthropic_content(message)
        content_blocks = apply_fallback_tool_detection(content_blocks, anthropic_tools)
        has_tool_use = any(b["type"] == "tool_use" for b in content_blocks)
        if has_tool_use:
            stop_reason = "tool_use"
        elif finish_reason == "length":
            stop_reason = "max_tokens"
        else:
            stop_reason = "end_turn"
        yield from emit_message_events(message_id, response_model, content_blocks, stop_reason)
        return

    # No tools in play: stream deltas through as they arrive (low latency).
    yield sse("message_start", {
        "type": "message_start",
        "message": {
            "id": message_id, "type": "message", "role": "assistant",
            "content": [], "model": response_model,
            "usage": {"input_tokens": 0, "output_tokens": 0},
        },
    })

    block_index = -1
    text_block_index = None
    finish_reason = None

    for line in resp.iter_lines():
        if not line:
            continue
        line_str = line.decode("utf-8", errors="ignore")
        if not line_str.startswith("data: "):
            continue
        data_str = line_str[6:]
        if data_str.strip() == "[DONE]":
            break
        try:
            chunk = json.loads(data_str)
        except json.JSONDecodeError:
            continue
        choices = chunk.get("choices") or []
        if not choices:
            continue
        choice = choices[0]
        delta = choice.get("delta", {})
        if choice.get("finish_reason"):
            finish_reason = choice.get("finish_reason")

        text_piece = delta.get("content")
        if text_piece:
            if text_block_index is None:
                block_index += 1
                text_block_index = block_index
                yield sse("content_block_start", {
                    "type": "content_block_start", "index": text_block_index,
                    "content_block": {"type": "text", "text": ""},
                })
            yield sse("content_block_delta", {
                "type": "content_block_delta", "index": text_block_index,
                "delta": {"type": "text_delta", "text": text_piece},
            })

    if text_block_index is not None:
        yield sse("content_block_stop", {"type": "content_block_stop", "index": text_block_index})

    stop_reason = "max_tokens" if finish_reason == "length" else "end_turn"
    yield sse("message_delta", {
        "type": "message_delta",
        "delta": {"stop_reason": stop_reason, "stop_sequence": None},
        "usage": {"output_tokens": 0},
    })
    yield sse("message_stop", {"type": "message_stop"})


@app.post("/v1/messages/count_tokens")
async def count_tokens(request: Request, _: None = Depends(verify_api_key)):
    try:
        body = await request.json()
    except Exception:
        return anthropic_error_response(400, "invalid JSON body", "invalid_request_error")

    total_text = system_to_text(body.get("system", ""))

    for msg in body.get("messages", []):
        content = msg.get("content")
        if isinstance(content, str):
            total_text += "\n" + content
        elif isinstance(content, list):
            for block in content:
                btype = block.get("type")
                if btype == "text":
                    total_text += "\n" + block.get("text", "")
                elif btype == "tool_result":
                    c = block.get("content", "")
                    total_text += "\n" + (c if isinstance(c, str) else json.dumps(c))
                elif btype == "tool_use":
                    total_text += "\n" + json.dumps(block.get("input", {}))

    for tool in body.get("tools", []) or []:
        total_text += "\n" + json.dumps(tool.get("input_schema", {}))

    return {"input_tokens": estimate_tokens(total_text)}


@app.get("/health")
def health_check():
    return {
        "status": "healthy",
        "model": MODEL_NAME,
        "endpoints": ["/chat", "/v1/chat/completions", "/v1/messages", "/v1/messages/count_tokens"],
    }


# ---------- start server in background thread ----------

import threading
import time as _time


def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning")


server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

for _ in range(30):
    try:
        r = requests.get("http://127.0.0.1:8000/health", timeout=2)
        if r.status_code == 200:
            break
    except requests.exceptions.RequestException:
        pass
    _time.sleep(1)
else:
    raise RuntimeError("FastAPI server did not become healthy in time")

print("FastAPI server running on port 8000")
print("Anthropic endpoint: /v1/messages (+ /v1/messages/count_tokens)")
print("OpenAI-compatible endpoints: /chat, /v1/chat/completions")
print("Health check: /health")
print(f"API key required via 'x-api-key' header: {API_KEY}")

## 8. cloudflared のインストール

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64

## 9. トンネルの起動

**注意**: このURLは再起動するたびに変わります。ノートブックを再実行した場合は必ず新しいURLを使ってください。

In [ ]:
import subprocess
import time
import re

subprocess.run(["pkill", "-9", "cloudflared"], stderr=subprocess.DEVNULL)

tunnel_process = subprocess.Popen(
    ["./cloudflared-linux-amd64", "tunnel", "--url", "http://localhost:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

print("Starting Cloudflare Tunnel...")
tunnel_url = None
deadline = time.time() + 30
while time.time() < deadline and tunnel_url is None:
    line = tunnel_process.stdout.readline()
    if not line:
        time.sleep(0.5)
        continue
    print(line.strip())
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        tunnel_url = match.group(0)

if not tunnel_url:
    raise RuntimeError("Could not find the tunnel URL in cloudflared's output - check the logs above")

print("\n" + "=" * 60)
print(f"Tunnel URL: {tunnel_url}")
print(f"API Key:    {API_KEY}")
print("=" * 60)
print("\nOn your local machine, run:")
print(f"  ./start-local-llm.sh {tunnel_url} {API_KEY}")
print("=" * 60)

## 10. 動作確認(ノートブック内・ローカルホスト経由)

トンネルを経由する前に、Colab環境内から直接サーバーを叩いて疎通確認します。
初回リクエストはモデルロードのため遅くなることがあります(数十秒〜)。

In [ ]:
import requests

r = requests.get("http://localhost:8000/health")
print("GET /health ->", r.status_code, r.json())

r = requests.post(
    "http://localhost:8000/v1/messages",
    headers={"x-api-key": API_KEY},
    json={
        "model": "claude-sonnet-4-5",
        "max_tokens": 200,
        "messages": [{"role": "user", "content": "Say hello in one short sentence."}],
    },
    timeout=180,
)
print("\nPOST /v1/messages ->", r.status_code)
print(r.json())